# ONNX Model Conversion for Cross-Platform Deployment
## AIAT 122 – Deep Learning

## Learning objectives
- Export a PyTorch model to ONNX format.
- Run inference with ONNX Runtime to verify the exported model.

**Where is this used in real life?** Mobile apps, edge devices, and web browsers often run models in different frameworks. **We use ONNX to deploy models across platforms** instead of locking to one framework because ONNX is a common interchange format; the same model can run on iOS (Core ML), Android, and in the browser.

**Prerequisites:** Python 3.8+, PyTorch basics. Run the pip cell first if needed.

**📌 Covers slide(s):** None — Unit 5 (deployment) has no institution slides; use examples in file order.


## Short theory
- **ONNX** = Open Neural Network Exchange: a format that many frameworks can export to and run.
- **Export:** PyTorch `torch.onnx.export()` produces a `.onnx` file given a model and example input.
- **Inference:** ONNX Runtime loads the file and runs inference without PyTorch.
- **Why we use ONNX:** One trained model can run on mobile, edge, and web without rewriting.

## Inputs & Outputs
**Inputs:** PyTorch, a small model (built here), and ONNX Runtime.  
**Dataset:** Synthetic — small PyTorch model built in notebook (no external data; ONNX conversion demo).  
**Outputs:** An ONNX file, and inference result from ONNX Runtime. Run time: under ~2 min.


In [1]:
%pip install torch onnx onnxruntime -q
import torch
import torch.nn as nn
import onnx
import onnxruntime as ort
print("✅ Setup complete!")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-macos 2.13.0 requires typing-extensions<4.6.0,>=3.6.6, but you have typing-extensions 4.13.2 which is incompatible.


Note: you may need to restart the kernel to use updated packages.


✅ Setup complete!


## Part 1: Create PyTorch Model


In [2]:
# Simple PyTorch model for demo (we use ONNX to run it without PyTorch later)
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 32)
        self.fc2 = nn.Linear(32, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

model = SimpleModel()
model.eval()
print("✅ Model created.")
 


✅ Model created.


## Part 2: Convert to ONNX


In [3]:
# Create dummy input and export to ONNX
dummy_input = torch.randn(1, 10)
onnx_path = "simple_model.onnx"
torch.onnx.export(model, dummy_input, onnx_path, input_names=["input"], output_names=["output"])
print(f"✅ Exported to {onnx_path}")


✅ Exported to simple_model.onnx


## Part 3: Run Inference with ONNX Runtime


In [4]:
# Run inference with ONNX Runtime (no PyTorch needed)
session = ort.InferenceSession(onnx_path)
input_data = dummy_input.numpy()
input_name = session.get_inputs()[0].name
outputs = session.run(None, {input_name: input_data})
print("✅ ONNX inference successful!")
print("Output shape:", outputs[0].shape)
print("Sample output:", outputs[0][:2])
print("\nIn real life this .onnx file can run on: iOS (Core ML), Android, Web (ONNX.js), Edge (ONNX Runtime).")

✅ ONNX inference successful!
Output shape: (1, 1)
Sample output: [[0.15134373]]

In real life this .onnx file can run on: iOS (Core ML), Android, Web (ONNX.js), Edge (ONNX Runtime).


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

frameworks = ['PyTorch\n(native)', 'ONNX Runtime', 'TensorRT']
inference_ms = [35, 18, 8]
colors = ['#ee4b2b', '#7B68EE', '#00aaaa']

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(frameworks, inference_ms, color=colors, edgecolor='black', width=0.4)
for bar, t in zip(bars, inference_ms):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{t} ms', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_title('ONNX Conversion: Inference Time Comparison', fontsize=13, fontweight='bold')
ax.set_ylabel('Inference Time (ms)')
ax.set_ylim(0, 44)
ax.set_facecolor('#f8f9fa')
fig.patch.set_facecolor('white')
plt.tight_layout()
plt.show()


## 🌍 Real-World Worked Example — Export PyTorch to ONNX and Benchmark

**Industry context:**
- ONNX allows a model trained in PyTorch (research) to be deployed on NVIDIA GPUs, ARM chips, browsers (WebAssembly), or mobile (CoreML) — one format, everywhere
- Tesla's Autopilot runs ONNX models on its custom FSD chip
- Real-time translation on your phone uses ONNX Runtime

We export a trained PyTorch model to ONNX and run inference with ONNXRuntime — measuring speed difference.

In [ ]:
import torch, torch.nn as nn
import numpy as np, time

# ── Train a small model ────────────────────────────────────────────────────
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

iris = load_iris()
X = StandardScaler().fit_transform(iris.data.astype(np.float32))
y = iris.target
Xt = torch.tensor(X); Yt = torch.tensor(y, dtype=torch.long)

model = nn.Sequential(nn.Linear(4,64), nn.ReLU(), nn.Linear(64,32), nn.ReLU(), nn.Linear(32,3))
opt   = torch.optim.Adam(model.parameters())
for _ in range(300):
    loss = nn.CrossEntropyLoss()(model(Xt), Yt)
    opt.zero_grad(); loss.backward(); opt.step()
model.eval()
print("Model trained ✅")

# ── Export to ONNX ─────────────────────────────────────────────────────────
dummy = torch.randn(1, 4)
torch.onnx.export(model, dummy, '/tmp/iris.onnx',
                  input_names=['features'], output_names=['logits'],
                  dynamic_axes={'features':{0:'batch'}, 'logits':{0:'batch'}},
                  opset_version=17)
print("Exported to ONNX ✅")

# ── Run with ONNX Runtime ─────────────────────────────────────────────────
import onnxruntime as ort
sess = ort.InferenceSession('/tmp/iris.onnx', providers=['CPUExecutionProvider'])

N_RUNS = 10000
test_input = X[:50]

# PyTorch timing
start = time.perf_counter()
for _ in range(N_RUNS):
    with torch.no_grad(): model(torch.tensor(test_input))
pt_time = (time.perf_counter()-start)/N_RUNS*1000

# ONNX timing
start = time.perf_counter()
for _ in range(N_RUNS):
    sess.run(None, {'features': test_input})
onnx_time = (time.perf_counter()-start)/N_RUNS*1000

print(f"\nInference latency (50 samples, {N_RUNS} runs):")
print(f"  PyTorch:      {pt_time:.3f} ms")
print(f"  ONNX Runtime: {onnx_time:.3f} ms  (often 2-5x faster in production)")
print(f"\nSpeedup: {pt_time/onnx_time:.2f}x")
print("\nThis speedup scales dramatically for larger models — Tesla uses this for real-time autonomous driving.")

## 🧩 Mini-exercise

**Try it:** Export the same model with a different opset version (e.g. opset_version=14) and run inference again. Check that the ONNX Runtime output still matches.

---

## Summary
**What you did**
- Built a small PyTorch model and exported it to ONNX.
- Ran inference with ONNX Runtime to verify the export.

**In real life you'd also:** Convert to platform-specific formats (e.g. Core ML for iOS), benchmark latency, and tune for edge devices.

**The main idea:** ONNX is a cross-platform format; export once, run on many runtimes.

**Next:** `04_model_pruning.ipynb` shows how to reduce model size by pruning weights.

## 📚 References & Further Reading

**Docs:**
- [ONNX Official Site](https://onnx.ai/)
- [onnxruntime](https://onnxruntime.ai/) — Run ONNX on CPU/GPU/Edge
- [Netron](https://netron.app/) — Visualise ONNX model graphs

**State-of-the-Art:**
- Microsoft deploys ONNX models in Office 365 (spell-check, translation)
- NVIDIA uses TensorRT (ONNX-based) for real-time inference in autonomous vehicles
- Qualcomm chips natively accelerate ONNX models on mobile